# Quantum Models in CLARYON

Demonstrates all quantum models: kernel SVM, simplified kernel SVM, Hadamard/SWAP distance classifiers, quantum GP, and QNN.

Uses a small Iris subset (4 features → 2 qubits) for fast execution.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize

# Binary Iris: setosa vs non-setosa, 4 features
iris = load_iris()
X = iris.data
y = (iris.target > 0).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Amplitude encode: pad to 2^2=4 (already 4 features), L2-normalize
from claryon.encoding.amplitude import amplitude_encode_matrix
X_train_q, enc_info = amplitude_encode_matrix(X_train)
X_test_q, _ = amplitude_encode_matrix(X_test, pad_len=enc_info.encoded_dim)
print(f"Encoded: {X_train_q.shape[1]} dims, {enc_info.n_qubits} qubits")

In [ ]:
from claryon.io.base import TaskType
from sklearn.metrics import balanced_accuracy_score

# Import all quantum models
from claryon.models.quantum.kernel_svm import QuantumKernelSVM
from claryon.models.quantum.sq_kernel_svm import SimplifiedQuantumKernelSVM
from claryon.models.quantum.qdc_hadamard import QuantumDistanceClassifierHadamard
from claryon.models.quantum.qdc_swap import QuantumDistanceClassifierSwap
from claryon.models.quantum.quantum_gp import QuantumGaussianProcess

n_qubits = enc_info.n_qubits

# Use small subsets for speed
X_tr_small = X_train_q[:20]
y_tr_small = y_train[:20]
X_te_small = X_test_q[:10]
y_te_small = y_test[:10]

models = {
    "kernel_svm": QuantumKernelSVM(n_qubits=n_qubits),
    "sq_kernel_svm": SimplifiedQuantumKernelSVM(n_qubits=n_qubits),
    "qdc_hadamard": QuantumDistanceClassifierHadamard(n_qubits=n_qubits),
    "qdc_swap": QuantumDistanceClassifierSwap(n_qubits=n_qubits),
    "quantum_gp": QuantumGaussianProcess(n_qubits=n_qubits, noise=0.4),
}

for name, model in models.items():
    model.fit(X_tr_small, y_tr_small, TaskType.BINARY)
    preds = model.predict(X_te_small)
    probs = model.predict_proba(X_te_small)
    bacc = balanced_accuracy_score(y_te_small, preds)
    print(f"{name:20s} BACC={bacc:.3f}  preds={preds.tolist()}")

In [ ]:
# QNN (slower — uses PyTorch training loop)
from claryon.models.quantum.qnn import QuantumNeuralNetwork

qnn = QuantumNeuralNetwork(n_qubits=n_qubits, n_layers=2, epochs=30, lr=0.01, batch_size=10)
qnn.fit(X_tr_small, y_tr_small, TaskType.BINARY)
preds = qnn.predict(X_te_small)
bacc = balanced_accuracy_score(y_te_small, preds)
print(f"{'qnn':20s} BACC={bacc:.3f}  preds={preds.tolist()}")

In [ ]:
# Geometric Difference score
from claryon.evaluation.geometric_difference import geometric_difference_score

# Compute quantum kernel matrix
model_ksvm = models["sq_kernel_svm"]
model_ksvm._init_pl()
K_Q = model_ksvm._kernel_matrix(X_tr_small, X_tr_small)

gdq = geometric_difference_score(K_Q, X_train=X_tr_small)
print(f"GDQ score: {gdq:.4f} (>1 suggests quantum advantage)")